# 01 - Bronze Ingestion

Reads raw JSON files using Structured Streaming and writes them to a Bronze Delta path with ingestion metadata and checkpointing.

In [ ]:
%run ../utilities/config

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("payment_method", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("source", StringType(), True),
])

bronze_stream = (
    spark.readStream
    .format("json")
    .schema(event_schema)
    .load(RAW_PATH)
)

bronze_df = (
    bronze_stream
    .withColumn("source_file", input_file_name())
    .withColumn("ingestion_timestamp", current_timestamp())
)

query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(availableNow=True)
    .start(BRONZE_PATH)
)

query.awaitTermination()

In [ ]:
display(spark.read.format("delta").load(BRONZE_PATH))